# Dataset & DataLoader

In [1]:
import os
import random

import numpy as np

import torch
from torch.utils.data import DataLoader, Subset
import torchvision.transforms as T

from dataset_fixed import TrainDataset, TestDataset

In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for this script because the submission block uses .cuda().")

device = torch.device("cuda")

In [3]:
image_size = 64
batch_size_train = 32
batch_size_eval = 32

mean = (0.485, 0.456, 0.406)
std  = (0.229, 0.224, 0.225)

val_ratio = 0.15
epochs = 15

# regularization
label_smoothing = 0.1
mixup_alpha = 0.2
mixup_prob = 0.5

save_path = "best_model_resnextse2.pth"

In [ ]:
# data augmentation
train_transform = T.Compose([
    T.Pad(4),
    T.RandomCrop(image_size),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(7),
    T.ColorJitter(brightness=0.05, contrast=0.05, saturation=0.05, hue=0.01),
    T.RandomGrayscale(p=0.05),
    T.ToTensor(),
    # RandomErasing operates on tensors
    T.Normalize(mean, std),
])

eval_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean, std),
])

In [5]:
data_root = "./cs441-assn3-data"

train_dataset_aug  = TrainDataset(root_path=data_root, transform=train_transform)
train_dataset_eval = TrainDataset(root_path=data_root, transform=eval_transform)
test_dataset = TestDataset(root_path=data_root, transform=eval_transform)

In [6]:
def stratified_split(labels: np.ndarray, val_ratio: float, seed: int):
    """Return (train_indices, val_indices) with per-class stratification.
    No sklearn dependency.
    """
    rng = np.random.default_rng(seed)
    labels = labels.astype(int)
    classes, counts = np.unique(labels, return_counts=True)

    val_indices = []
    train_indices = []

    for c, cnt in zip(classes, counts):
        idx = np.where(labels == c)[0]
        rng.shuffle(idx)
        n_val_c = int(round(cnt * val_ratio))
        # keep at least 1 sample in train if possible
        n_val_c = min(max(n_val_c, 1), max(cnt - 1, 1))
        val_indices.extend(idx[:n_val_c].tolist())
        train_indices.extend(idx[n_val_c:].tolist())

    rng.shuffle(train_indices)
    rng.shuffle(val_indices)
    return train_indices, val_indices

In [7]:
labels = np.array(train_dataset_eval.labels, dtype=int)
train_indices, val_indices = stratified_split(labels, val_ratio=val_ratio, seed=SEED)

train_subset = Subset(train_dataset_aug, train_indices)
val_subset   = Subset(train_dataset_eval, val_indices)

num_workers = min(8, os.cpu_count() or 4)

train_loader = DataLoader(
    train_subset,
    batch_size=batch_size_train,
    shuffle=True,
    num_workers=num_workers,
    drop_last=False,
    pin_memory=True,
    persistent_workers=(num_workers > 0),
)

val_loader = DataLoader(
    val_subset,
    batch_size=batch_size_eval,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=(num_workers > 0),
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size_eval,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=(num_workers > 0),
)

# Your Awesome Model

In [8]:
import torch
import torch.nn as nn

from resnext_fixed import SEResNeXt50

In [9]:
class SEResNeXtTTA50(nn.Module):
    def __init__(self, num_classes=15):
        super().__init__()
        self.resnext = SEResNeXt50(
            num_classes=num_classes,
            cardinality=32,
            base_width=12,
            drop_path_rate=0.10,
        )

    def forward(self, x):
        if self.training:
            return self.resnext(x)

        # eval-time TTA (hflip)
        x_flipped = torch.flip(x, dims=[-1])
        out_orig = self.resnext(x)
        out_flip = self.resnext(x_flipped)
        return (out_orig + out_flip) * 0.5

In [10]:
model = SEResNeXtTTA50(num_classes=15).to(device)

# Model parameter checking

In [11]:
# model parameters checking
num_params = sum(p.numel() for p in model.parameters())
print('The number of your model parameters :', num_params)
print('Parameter usage : ' + str(num_params/1000000) + '%')
if num_params > 100000000:
  raise Exception('Compress your model.')

The number of your model parameters : 74349055
Parameter usage : 74.349055%


# Model training

In [12]:
import tqdm

import torch
import torch.nn as nn
import torch.optim as optim

from torch.optim.lr_scheduler import SequentialLR, LambdaLR, CosineAnnealingLR

In [13]:
print("GPU count:", torch.cuda.device_count())
print("Current device index:", torch.cuda.current_device())
print("Current device name:", torch.cuda.get_device_name(torch.cuda.current_device()))

GPU count: 1
Current device index: 0
Current device name: NVIDIA GeForce RTX 4080 SUPER


In [14]:
def mixup_batch(x, y, alpha: float):
    """Return mixed inputs and two targets for mixup CE."""
    if alpha <= 0:
        return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)
    mixed_x = lam * x + (1.0 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

In [15]:
criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing).to(device)
optimizer = optim.AdamW(model.parameters(), lr=4e-4, weight_decay=5e-4)

scaler = torch.amp.GradScaler("cuda")

total_steps = epochs * len(train_loader)
warmup_steps = max(1, int(0.1 * total_steps))
T_max_cosine = max(1, total_steps - warmup_steps)

scheduler = SequentialLR(
    optimizer,
    schedulers=[
        LambdaLR(optimizer, lambda step: (step + 1) / warmup_steps),
        CosineAnnealingLR(optimizer, T_max=T_max_cosine),
    ],
    milestones=[warmup_steps],
)

best_val_acc = -1.0

In [16]:
print("Start Training Model...")

for epoch in range(epochs):
    # TRAIN
    model.train()
    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for x, y in tqdm.tqdm(train_loader, desc=f"Epoch {epoch} [Train]"):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        # optional mixup
        if mixup_alpha > 0 and np.random.rand() < mixup_prob:
            x_m, y_a, y_b, lam = mixup_batch(x, y, mixup_alpha)
            with torch.amp.autocast("cuda"):
                out = model(x_m)
                loss = lam * criterion(out, y_a) + (1.0 - lam) * criterion(out, y_b)
        else:
            with torch.amp.autocast("cuda"):
                out = model(x)
                loss = criterion(out, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        # stats (use non-mixed y for accuracy; mixup accuracy isn't meaningful)
        train_loss_sum += loss.item() * y.size(0)
        train_correct += (out.argmax(1) == y).sum().item()
        train_total += y.size(0)

    train_loss = train_loss_sum / max(1, train_total)
    train_acc = train_correct / max(1, train_total)
    print(f"Epoch {epoch} | Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f}")

    # VALIDATION
    model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for x, y in tqdm.tqdm(val_loader, desc=f"Epoch {epoch} [Val]"):
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            out = model(x)
            loss = criterion(out, y)

            val_loss_sum += loss.item() * y.size(0)
            val_correct += (out.argmax(1) == y).sum().item()
            val_total += y.size(0)

    val_loss = val_loss_sum / max(1, val_total)
    val_acc = val_correct / max(1, val_total)
    print(f"Epoch {epoch} | Val Loss: {val_loss:.4f} | Acc: {val_acc:.4f}")

    # checkpointing
    current_save_path = f"model_resnextse2_{epoch:02d}.pth"
    torch.save(model.state_dict(), current_save_path)
    print(f"Saved Model for Epoch {epoch} at {current_save_path}")

    # choose best by validation accuracy (matches typical competition metric)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), save_path)
        print(f"Saved Best Model (Val Acc: {val_acc:.4f})")

Start Training Model...


Epoch 0 [Train]: 100%|██████████| 1196/1196 [10:08<00:00,  1.96it/s]


Epoch 0 | Train Loss: 2.5201 | Acc: 0.1733


Epoch 0 [Val]: 100%|██████████| 211/211 [01:45<00:00,  1.99it/s]


Epoch 0 | Val Loss: 2.3218 | Acc: 0.2908
Saved Model for Epoch 0 at model_resnextse2_00.pth
Saved Best Model (Val Acc: 0.2908)


...

Epoch 13 | Val Loss: 1.1963 | Acc: 0.7456
Saved Model for Epoch 13 at model_resnextse2_13.pth
Saved Best Model (Val Acc: 0.7456)


Epoch 14 [Train]: 100%|██████████| 1196/1196 [12:53<00:00,  1.55it/s]


Epoch 14 | Train Loss: 1.3459 | Acc: 0.5699


Epoch 14 [Val]: 100%|██████████| 211/211 [06:13<00:00,  1.77s/it]


Epoch 14 | Val Loss: 1.1961 | Acc: 0.7436
Saved Model for Epoch 14 at model_resnextse2_14.pth


In [17]:
model.load_state_dict(torch.load(save_path, map_location=device))

C:\Users\user\AppData\Local\Temp\ipykernel_6368\1058478286.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(save_path, map_location=devic

<All keys matched successfully>

# Submit
Do not edit the submission code below.

In [18]:
import pandas as pd

# Load Best Model
submit = pd.read_csv('./cs441-assn3-data/Test_64.csv')

# model parameters checking
num_params = sum(p.numel() for p in model.parameters())
print('The number of your model parameters :', num_params)
print('Parameter usage : ' + str(num_params/1000000) + '%')
if num_params > 100000000:
  raise Exception('Compress your model.')

total_prediction = list()
model.eval()
with torch.no_grad():
    for x in tqdm.tqdm(test_loader):
        x = torch.FloatTensor(x).cuda()
        output = model(x)
        predict = torch.argmax(output,dim=1)
        total_prediction.extend(predict.cpu().numpy())
    submit['label'] = total_prediction
    submit.to_csv('submission.csv',index=False)

The number of your model parameters : 74349055
Parameter usage : 74.349055%


100%|██████████| 235/235 [07:16<00:00,  1.86s/it]
